# Data Exploration Notebook

## Zomato Restaurant Recommendation System - Phase 1

This notebook explores the processed Zomato restaurant dataset to understand its characteristics and prepare for building the recommendation system.

### 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

### 2. Load Processed Dataset

In [ ]:
# Load the processed CSV file
data_path = Path('./processed_data/zomato_restaurants_processed.csv')

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("Processed data file not found. Please run preprocess_data.py first.")

### 3. Basic Dataset Overview

In [ ]:
# Display basic information
print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
print("\n" + "="*60)
print("FIRST 5 RECORDS")
print("="*60)
display(df.head())

### 4. Data Types and Missing Values

In [ ]:
# Check data types
print("="*60)
print("DATA TYPES")
print("="*60)
print(df.dtypes)

# Check for missing values
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values found!")

### 5. Statistical Summary

In [ ]:
# Display statistical summary for numeric columns
print("="*60)
print("STATISTICAL SUMMARY")
print("="*60)
numeric_cols = df.select_dtypes(include=[np.number]).columns
display(df[numeric_cols].describe())

### 6. Location Analysis

In [ ]:
# Analyze city distribution
if 'city' in df.columns:
    city_counts = df['city'].value_counts().head(15)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=city_counts.values, y=city_counts.index, palette='viridis')
    plt.title('Top 15 Cities by Restaurant Count', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Restaurants')
    plt.ylabel('City')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal unique cities: {df['city'].nunique()}")
    print(f"\nTop 10 cities by restaurant count:")
    print(city_counts.head(10))

### 7. Cost Analysis

In [ ]:
if 'cost' in df.columns:
    # Cost distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df['cost'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_title('Cost Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Cost for Two (INR)')
    axes[0].set_ylabel('Frequency')
    
    # Box plot
    axes[1].boxplot(df['cost'], vert=False)
    axes[1].set_title('Cost Box Plot', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Cost for Two (INR)')
    
    plt.tight_layout()
    plt.show()
    
    # Cost statistics
    print(f"\nCost Statistics:")
    print(f"  Minimum: ₹{df['cost'].min():,.0f}")
    print(f"  Maximum: ₹{df['cost'].max():,.0f}")
    print(f"  Mean: ₹{df['cost'].mean():,.0f}")
    print(f"  Median: ₹{df['cost'].median():,.0f}")
    print(f"  25th Percentile: ₹{df['cost'].quantile(0.25):,.0f}")
    print(f"  75th Percentile: ₹{df['cost'].quantile(0.75):,.0f}")

### 8. Rating Analysis

In [ ]:
if 'rating' in df.columns:
    # Rating distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df['rating'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].set_title('Rating Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Rating (0-5)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_xlim(0, 5)
    
    # Count plot for rating categories
    if 'rating_category' in df.columns:
        sns.countplot(data=df, x='rating_category', ax=axes[1], palette='Set2')
        axes[1].set_title('Rating Categories', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Category')
        axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    # Rating statistics
    print(f"\nRating Statistics:")
    print(f"  Minimum: {df['rating'].min():.1f}")
    print(f"  Maximum: {df['rating'].max():.1f}")
    print(f"  Mean: {df['rating'].mean():.2f}")
    print(f"  Median: {df['rating'].median():.2f}")
    print(f"  Standard Deviation: {df['rating'].std():.2f}")

### 9. Cuisine Analysis

In [ ]:
# Analyze cuisine distribution
if 'cuisine_list' in df.columns:
    # Flatten all cuisines
    all_cuisines = []
    for cuisine_list in df['cuisine_list']:
        if isinstance(cuisine_list, str):
            # Parse if stored as string
            try:
                cuisines = eval(cuisine_list)
                all_cuisines.extend(cuisines)
            except:
                all_cuisines.append(cuisine_list)
        elif isinstance(cuisine_list, list):
            all_cuisines.extend(cuisine_list)
    
    # Count cuisine occurrences
    from collections import Counter
    cuisine_counts = Counter(all_cuisines)
    
    # Get top 20 cuisines
    top_cuisines = cuisine_counts.most_common(20)
    cuisines, counts = zip(*top_cuisines)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=list(counts), y=list(cuisines), palette='rocket')
    plt.title('Top 20 Cuisines', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Restaurants')
    plt.ylabel('Cuisine')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal unique cuisines: {len(cuisine_counts)}")
    print(f"\nTop 15 cuisines:")
    for cuisine, count in top_cuisines[:15]:
        print(f"  {cuisine}: {count}")

### 10. Budget Category Analysis

In [ ]:
if 'budget_category' in df.columns:
    # Budget distribution
    budget_counts = df['budget_category'].value_counts()
    
    plt.figure(figsize=(8, 6))
    colors = ['#66c2a5', '#fc8d62', '#8da0cb']
    plt.pie(budget_counts.values, labels=budget_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
    plt.title('Budget Category Distribution', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.show()
    
    print(f"\nBudget Category Counts:")
    for category, count in budget_counts.items():
        print(f"  {category}: {count} ({count/len(df)*100:.1f}%)")

### 11. Cost vs Rating Analysis

In [ ]:
if 'cost' in df.columns and 'rating' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='cost', y='rating', alpha=0.5, s=50)
    plt.title('Cost vs Rating', fontsize=14, fontweight='bold')
    plt.xlabel('Cost for Two (INR)')
    plt.ylabel('Rating')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Calculate correlation
    correlation = df['cost'].corr(df['rating'])
    print(f"\nCorrelation between Cost and Rating: {correlation:.3f}")

### 12. City-wise Average Cost and Rating

In [ ]:
if 'city' in df.columns and 'cost' in df.columns and 'rating' in df.columns:
    # Calculate city-wise statistics
    city_stats = df.groupby('city').agg({
        'cost': ['mean', 'median'],
        'rating': 'mean',
        'name': 'count'
    }).round(2)
    
    city_stats.columns = ['Avg Cost', 'Median Cost', 'Avg Rating', 'Restaurant Count']
    city_stats = city_stats.sort_values('Restaurant Count', ascending=False).head(15)
    
    print("\nCity-wise Statistics (Top 15 by Restaurant Count):")
    print("="*80)
    display(city_stats)

### 13. Sample Query: Filter Restaurants

In [ ]:
# Example: Find restaurants in Bangalore with cost <= 1000 and rating >= 4.0
sample_query = df[
    (df['city'] == 'Bangalore') &
    (df['cost'] <= 1000) &
    (df['rating'] >= 4.0)
].sort_values('rating', ascending=False).head(10)

print("Sample Query Results: Bangalore restaurants (Cost ≤ ₹1000, Rating ≥ 4.0)")
print("="*80)
print(f"Found {len(sample_query)} restaurants")
display(sample_query[['name', 'location', 'cuisines', 'cost', 'rating']])

### 14. Data Quality Summary

In [ ]:
print("="*80)
print("DATA QUALITY SUMMARY")
print("="*80)
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Records: {df.duplicated().sum()}")
print(f"\nUnique Values:")
print(f"  Cities: {df['city'].nunique() if 'city' in df.columns else 'N/A'}")
print(f"  Locations: {df['location'].nunique() if 'location' in df.columns else 'N/A'}")
print(f"  Cuisines: {df['cuisines'].nunique() if 'cuisines' in df.columns else 'N/A'}")
print(f"\nNumeric Ranges:")
if 'cost' in df.columns:
    print(f"  Cost: ₹{df['cost'].min():,.0f} - ₹{df['cost'].max():,.0f}")
if 'rating' in df.columns:
    print(f"  Rating: {df['rating'].min():.1f} - {df['rating'].max():.1f}")
print("\n" + "="*80)
print("✓ Data quality check complete!")
print("="*80)

### 15. Export Summary Statistics

In [ ]:
# Create summary statistics dictionary
summary = {
    'total_records': int(len(df)),
    'total_columns': int(len(df.columns)),
    'missing_values': int(df.isnull().sum().sum()),
    'duplicate_records': int(df.duplicated().sum()),
    'unique_cities': int(df['city'].nunique()) if 'city' in df.columns else 0,
    'unique_locations': int(df['location'].nunique()) if 'location' in df.columns else 0,
    'unique_cuisines': int(df['cuisines'].nunique()) if 'cuisines' in df.columns else 0,
    'cost_range': {
        'min': float(df['cost'].min()) if 'cost' in df.columns else 0,
        'max': float(df['cost'].max()) if 'cost' in df.columns else 0,
        'mean': float(df['cost'].mean()) if 'cost' in df.columns else 0,
        'median': float(df['cost'].median()) if 'cost' in df.columns else 0
    },
    'rating_range': {
        'min': float(df['rating'].min()) if 'rating' in df.columns else 0,
        'max': float(df['rating'].max()) if 'rating' in df.columns else 0,
        'mean': float(df['rating'].mean()) if 'rating' in df.columns else 0,
        'median': float(df['rating'].median()) if 'rating' in df.columns else 0
    }
}

# Save summary to JSON
with open('./processed_data/exploration_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Summary statistics exported to: processed_data/exploration_summary.json")

## Conclusion

This notebook provides a comprehensive exploration of the processed Zomato restaurant dataset. Key insights:

1. **Data Quality**: The dataset has been cleaned and preprocessed with no missing values in critical fields
2. **Geographic Distribution**: Restaurants are distributed across multiple cities with varying densities
3. **Cost Distribution**: Wide range of costs suitable for different budget preferences
4. **Rating Distribution**: Most restaurants have good ratings, indicating quality data
5. **Cuisine Diversity**: Rich variety of cuisines available across different locations

This analysis confirms the dataset is ready for building the recommendation system.